# Credit Card Fraud Detection — MLflow Experiment Tracking

Track the final LightGBM + SMOTE configuration and its test-set metrics in MLflow.

## Experiment Configuration

The tracked configuration is the Stage 3 winning model with the Stage 4 operating threshold.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import mlflow
import mlflow.sklearn
import pandas as pd

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split

from src.features import add_time_features, add_training_volume_feature
from src.preprocessing import make_preprocessor


In [2]:
DATA_PATH = PROJECT_ROOT / "data" / "creditcard.csv"

RANDOM_STATE = 42
TEST_SIZE = 0.20
SMOTE_SAMPLING_STRATEGY = 0.10
SELECTED_THRESHOLD = 0.239
FP_COST = 1
FN_COST = 10

LGBM_PARAMS = {
    "objective": "binary",
    "n_estimators": 300,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": -1,
}

print("Data path:", DATA_PATH)


Data path: d:\3\JupyterProjects\ML Projects\fraud-detection\data\creditcard.csv


In [3]:
df = pd.read_csv(DATA_PATH)

X = df.drop(columns="Class")
y = df["Class"]

print("Dataset shape:", df.shape)
print("Fraud transactions:", int(y.sum()))
print(f"Fraud rate: {y.mean() * 100:.4f}%")

Dataset shape: (284807, 31)
Fraud transactions: 492
Fraud rate: 0.1727%


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_train = add_time_features(X_train)
X_test = add_time_features(X_test)

X_train, X_test, hourly_counts = add_training_volume_feature(
    X_train,
    X_test,
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))
print("Features after engineering:", X_train.shape[1])
print("Hourly buckets learned:", len(hourly_counts))

Training samples: 227845
Test samples: 56962
Features after engineering: 35
Hourly buckets learned: 48


In [5]:
preprocessor = make_preprocessor()

smote = SMOTE(
    sampling_strategy=SMOTE_SAMPLING_STRATEGY,
    random_state=RANDOM_STATE,
)

model = LGBMClassifier(**LGBM_PARAMS)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("smote", smote),
        ("model", model),
    ]
)

print(pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('robust_scaler',
                                                  RobustScaler(),
                                                  ['Time', 'Amount'])],
                                   verbose_feature_names_out=False)),
                ('smote', SMOTE(random_state=42, sampling_strategy=0.1)),
                ('model',
                 LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05,
                                n_estimators=300, n_jobs=-1, objective='binary',
                                random_state=42, subsample=0.8,
                                verbosity=-1))])


## MLflow Tracking

Use a local SQLite tracking database at the project root.

In [6]:
MLFLOW_DB_PATH = PROJECT_ROOT / "mlflow.db"

mlflow.set_tracking_uri(
    f"sqlite:///{MLFLOW_DB_PATH.as_posix()}"
)

mlflow.set_experiment(
    "credit-card-fraud-detection"
)

print("MLflow tracking URI:", mlflow.get_tracking_uri())

print(
    "Experiment:",
    mlflow.get_experiment_by_name(
        "credit-card-fraud-detection"
    ).name
)

MLflow tracking URI: sqlite:///d:/3/JupyterProjects/ML Projects/fraud-detection/mlflow.db
Experiment: credit-card-fraud-detection


In [7]:
with mlflow.start_run(run_name="LightGBM_SMOTE_final"):
    mlflow.log_params({
        "model": "LightGBM",
        "imbalance_strategy": "SMOTE",
        "smote_sampling_strategy": SMOTE_SAMPLING_STRATEGY,
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "decision_threshold": SELECTED_THRESHOLD,
        "false_positive_cost": FP_COST,
        "false_negative_cost": FN_COST,
    })

    for name, value in LGBM_PARAMS.items():
        mlflow.log_param(f"lgbm_{name}", value)

    pipeline.fit(X_train, y_train)

    test_probabilities = pipeline.predict_proba(X_test)[:, 1]
    test_predictions = (test_probabilities >= SELECTED_THRESHOLD).astype(int)

    pr_auc = average_precision_score(y_test, test_probabilities)
    roc_auc = roc_auc_score(y_test, test_probabilities)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        test_predictions,
        labels=[0, 1],
    ).ravel()

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
    financial_cost = fp * FP_COST + fn * FN_COST

    mlflow.log_metrics({
        "test_pr_auc": pr_auc,
        "test_roc_auc": roc_auc,
        "test_precision": precision,
        "test_recall": recall,
        "test_f1": f1,
        "test_financial_cost": financial_cost,
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "true_negatives": tn,
    })

    mlflow.sklearn.log_model(
        pipeline,
        name="fraud_detection_pipeline",
        skops_trusted_types=[
            "collections.OrderedDict",
            "imblearn.over_sampling._smote.base.SMOTE",
            "imblearn.pipeline.Pipeline",
            "lightgbm.basic.Booster",
            "lightgbm.sklearn.LGBMClassifier",
        ],
    )

    run_id = mlflow.active_run().info.run_id

    print("MLflow run completed")
    print("Run ID:", run_id)
    print(f"PR-AUC:          {pr_auc:.4f}")
    print(f"ROC-AUC:         {roc_auc:.4f}")
    print(f"Precision:       {precision:.4f}")
    print(f"Recall:          {recall:.4f}")
    print(f"F1:              {f1:.4f}")
    print(f"Financial cost:  {financial_cost:.0f}")


2026/09/10 22:23:43 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


MLflow run completed
Run ID: c9a29456a2ae431d8776a5b4c13f7fa7
PR-AUC:          0.8744
ROC-AUC:         0.9793
Precision:       0.7699
Recall:          0.8878
F1:              0.8246
Financial cost:  136


In [8]:
# Verify the run

experiment = mlflow.get_experiment_by_name("credit-card-fraud-detection")

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.test_pr_auc DESC"],
)

display(
    runs[
        [
            "run_id",
            "status",
            "params.model",
            "params.imbalance_strategy",
            "params.decision_threshold",
            "metrics.test_pr_auc",
            "metrics.test_roc_auc",
            "metrics.test_precision",
            "metrics.test_recall",
            "metrics.test_f1",
            "metrics.test_financial_cost",
        ]
    ]
)


,run_id,status,params.model,params.imbalance_strategy,params.decision_threshold,metrics.test_pr_auc,metrics.test_roc_auc,metrics.test_precision,metrics.test_recall,metrics.test_f1,metrics.test_financial_cost
0,c9a29456a2ae431d8776a5b4c13f7fa7,FINISHED,LightGBM,SMOTE,0.239,0.874444,0.979296,0.769912,0.887755,0.824645,136.0
1,3dd60d3415954f9ebed442fa41e2dd64,FINISHED,LightGBM,SMOTE,0.239,0.874444,0.979296,0.769912,0.887755,0.824645,136.0
2,788d41da31104883a832b0c1a7196440,FAILED,LightGBM,SMOTE,0.239,0.874444,0.979296,0.769912,0.887755,0.824645,136.0


## Expected Results

The tracked run should be close to the final production training result. Small differences can occur across dependency versions.

| Metric | Reference |
|---|---:|
| PR-AUC | 0.8744 |
| ROC-AUC | 0.9793 |
| Precision | 0.7699 |
| Recall | 0.8878 |
| F1 | 0.8246 |
| Financial cost | 136 |